## Polymorphism in Python: One Interface, Many Behaviors

**Goal:** learn to write code that sends the same message to different objects and lets each object respond in its own way.

Work through each section in order. For each code cell, **predict** what will happen, run it, then explain the result in your own words. Exercise cells are intentionally incomplete; try them before opening the answer key at the end.

**You’ll practice:** shared methods, duck typing, method overriding, operator overloading, built-in polymorphism, and a small design challenge.

## 1. The core idea

*Polymorphism* means _many forms_. In a program, it means the same operation can produce different behavior depending on the object receiving it.

Imagine a function that asks an animal to make a sound. The function does not need to know whether it received a bird or a dog. It only needs the object to provide the behavior it asks for.

In [ ]:
class Bird:
    def sound(self):
        return "Tweet"

class Dog:
    def sound(self):
        return "Bark"

def make_sound(animal):
    return animal.sound()

bird = Bird()
dog = Dog()

print(make_sound(bird))
print(make_sound(dog))

**Notice:** `make_sound` is written once. The result changes because each object supplies its own `sound()` method.

### Quick check

1. Which line chooses whether the result is `Tweet` or `Bark`?
2. Does `make_sound` need an `if` statement that checks the object's class?


## 2. Duck typing: behavior matters

Python often cares more about what an object can do than what class it belongs to. This is called **duck typing**: if an object has the method your code needs, it can take part.

In [1]:
class Duck:
    def fly(self):
        return "Duck flying"

class Airplane:
    def fly(self):
        return "Airplane flying"

def test_flight(thing):
    return thing.fly()

for thing in [Duck(), Airplane()]:
    print(test_flight(thing))

Duck flying
Airplane flying


The function asks for one capability: `fly()`. It does not require a shared parent class.

### Your turn: add a flying thing

Create a `Kite` class with a `fly()` method that returns `"Kite gliding"`. Then call `test_flight(Kite())`.

In [ ]:
# TODO: define Kite, then try it with test_flight


### When an object lacks the expected behavior

Duck typing is flexible, but the object still has to meet the expectation. Try the next cell and read the error.

In [4]:
class Rock:
    pass

# Uncomment to see what happens:
# test_flight(Rock())

The error is Python telling us that `Rock` does not provide the `fly()` behavior this function needs.

## 3. Inheritance and method overriding

A child class can inherit a method and replace it with a more specific version. This is **method overriding**. Each subclass still supports the shared method name, so other code can work with them uniformly.

In [ ]:
class Animal:
    def speak(self):
        return "Some animal sound"

class Cat(Animal):
    def speak(self):
        return "Meow"

class Cow(Animal):
    def speak(self):
        return "Moo"

for animal in [Animal(), Cat(), Cow()]:
    print(animal.speak())

### Your turn: add another subclass

Create `Sheep`, inheriting from `Animal`, and override `speak()` to return `"Baa"`. Add a `Sheep()` to the list and run the loop again.

In [ ]:
# TODO: define Sheep and include it in this list
# animals = [Animal(), Cat(), Cow()]


### Inheritance is not required for polymorphism

`Bird` and `Dog` in section 1 did not inherit from a shared class, but the same function still worked with both. Shared behavior can come from inheritance, or simply from objects following the same informal interface.

## 4. One loop, many shapes

A common use for polymorphism is processing a collection of different objects with the same code. Each shape knows how to calculate its own area and perimeter.

In [ ]:
import math

class Rectangle:
    def __init__(self, length, width):
        self.length = length
        self.width = width

    def area(self):
        return self.length * self.width

    def perimeter(self):
        return 2 * (self.length + self.width)

class Circle:
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return math.pi * self.radius ** 2

    def perimeter(self):
        return 2 * math.pi * self.radius

shapes = [Rectangle(5, 4), Circle(3)]
for shape in shapes:
    print(f"{type(shape).__name__}: area={shape.area():.2f}, perimeter={shape.perimeter():.2f}")

The loop does not need to know each shape’s formula. It relies on the shared `area()` and `perimeter()` interface.

### Your turn: make a triangle shape

Add a `Square` class with one `side` attribute and the same two methods. Add `Square(4)` to `shapes`. Confirm the loop works without changing it.

In [ ]:
# TODO: define Square(side) with area() and perimeter()
# Then add Square(4) to shapes and rerun the loop above.


## 5. Polymorphism in operators

Python operators are also interfaces. `+` adds numbers, joins strings, and combines lists. A class can define what `+` means for its objects by implementing `__add__`.

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Point(self.x + other.x, self.y + other.y)

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

p1 = Point(1, 2)
p2 = Point(3, 4)
print(p1 + p2)
print(2 + 3)
print("Py" + "thon")

### Your turn: add point subtraction

Implement `__sub__` on `Point` so `Point(8, 7) - Point(2, 3)` gives `Point(6, 4)`. Add the method to the class above and try the expression.

In [ ]:
# TODO: add this method inside Point:
# def __sub__(self, other):
#     return Point(self.x - other.x, self.y - other.y)


## 6. Built-in functions use common interfaces

`len()` works with strings, lists, and dictionaries because each type knows how to report its length. `str()` asks different objects for a readable string representation.

In [ ]:
items = ["hello", [1, 2, 3], {"name": "Ada", "age": 36}]
for item in items:
    print(type(item).__name__, "length:", len(item))

for value in [123, [1, 2, 3], {"a": 1}]:
    print(str(value))

This is the same broad idea: callers use one operation, and objects provide type-appropriate behavior.

## 7. A flexible function is related, but different

Python does not support traditional method overloading by defining several functions with the same name and different parameter lists. A function can still handle different argument patterns using defaults. This is related to flexible interfaces, though it is not the same mechanism as objects overriding a method.

In [ ]:
def calculate_area(length, width=None):
    if width is None:
        return length * length  # square
    return length * width       # rectangle

print(calculate_area(5))
print(calculate_area(4, 6))

### Your turn: support a circle

Extend `calculate_area` with a keyword-only `radius` option, using `math.pi * radius ** 2`. Keep the existing square and rectangle behavior. Try `calculate_area(radius=2)`.

*Hint:* you may want to validate that the caller supplies exactly one shape description.

In [ ]:
# TODO: write your extended function here


## 8. Design challenge: notification senders

A notification service should send messages through different channels. Each channel has a `send(message)` method, but the service does not need to know how sending works.

First, run the sample.

In [ ]:
class EmailSender:
    def send(self, message):
        return f"Email sent: {message}"

class TextSender:
    def send(self, message):
        return f"Text sent: {message}"

def notify(sender, message):
    return sender.send(message)

for sender in [EmailSender(), TextSender()]:
    print(notify(sender, "Your order is ready"))

### Your turn

1. Add a `PushSender` that returns `"Push notification sent: ..."`.
2. Use the same `notify` function with it.
3. In one or two sentences, explain why adding a new sender did not require editing `notify`.

Optional: change `notify` to accept a list of senders and send the message through each one.

In [ ]:
# TODO: define PushSender and try notify(PushSender(), "Hello!")


## 9. Common design trap: checking every type

If code grows into a long chain of class checks, it often becomes harder to extend. Compare the branching version below with the polymorphic `notify` function above.

In [ ]:
def notify_with_type_checks(sender, message):
    if isinstance(sender, EmailSender):
        return f"Email sent: {message}"
    elif isinstance(sender, TextSender):
        return f"Text sent: {message}"
    else:
        raise TypeError("Unsupported sender")

print(notify_with_type_checks(EmailSender(), "Hello"))

With the type-checking version, every new sender may require another branch. With the polymorphic version, each sender provides `send()` and the caller stays the same. Type checks can still be useful, but behavior-based interfaces are often simpler when several types support the same operation.

## 10. Review

Complete these prompts before checking the answer key:

1. Explain polymorphism in your own words.
2. What does duck typing look for?
3. How does overriding differ from duck typing?
4. What special method controls `+` for a custom class?
5. What common interface did `Rectangle` and `Circle` share?
6. Name one place you might use polymorphism in a real program.

## Answer key

Try the exercises before reading this section.

### Section 2: Kite

```python
class Kite:
    def fly(self):
        return "Kite gliding"

test_flight(Kite())
```

### Section 3: Sheep

```python
class Sheep(Animal):
    def speak(self):
        return "Baa"

for animal in [Animal(), Cat(), Cow(), Sheep()]:
    print(animal.speak())
```

### Section 4: Square

```python
class Square:
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2

    def perimeter(self):
        return 4 * self.side

shapes.append(Square(4))
```

### Section 5: subtraction

```python
def __sub__(self, other):
    return Point(self.x - other.x, self.y - other.y)
```

Put the method inside `Point` next to `__add__`.

### Section 7: area with a circle option

```python
def calculate_area(length=None, width=None, *, radius=None):
    if radius is not None and length is None and width is None:
        return math.pi * radius ** 2
    if radius is None and length is not None and width is None:
        return length ** 2
    if radius is None and length is not None and width is not None:
        return length * width
    raise ValueError("Provide a radius, a side length, or length and width")
```

Other designs are possible; the key is to handle the supported cases clearly.

### Section 8: Push sender

```python
class PushSender:
    def send(self, message):
        return f"Push notification sent: {message}"

notify(PushSender(), "Hello!")
```

`notify` only relies on the `send()` behavior, so it works with any sender that supplies that method.

### Review prompts

1. One operation or interface can lead to different behavior for different objects.
2. The behavior an object supports (such as a method), rather than its exact class.
3. Duck typing needs compatible behavior; overriding is a subclass replacing a parent method implementation.
4. `__add__`.
5. Both had `area()` and `perimeter()`.
6. Examples include notification channels, file handlers, GUI controls, or database adapters.